# 01 Data Cleaning

This notebook sets up the initial cleaned datasets for the project using the workbook `annual-energy-consumption-data-2024_cleaning.xlsx`.

For now, the notebook does four things:
1. Loads the Excel workbook with pandas.
2. Creates a `properties` dataframe from the `Properties` sheet using only the required columns.
3. Creates separate gas and electric meter-entry dataframes using only the required columns.
4. Merges the property metadata into the gas and electric dataframes using `Property Name` and `Portfolio Manager ID`.

## Import libraries and define the file path

This cell imports pandas, sets the workbook path, and confirms the available sheet names before we start selecting columns.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

file_path = Path("../data/clean/annual-energy-consumption-data-2024_cleaning.xlsx")

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['Properties', 'Meter Entries - Gas', 'Meter Entries - Electeric']

## Create the `properties` dataframe

From the first worksheet, `Properties`, we keep only these columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `H`: `Property Type - Self-Selected`
- `I`: `Gross Floor Area`

These are the property-level fields needed later for analysis and modeling.

In [3]:
properties = pd.read_excel(
    file_path,
    sheet_name="Properties",
    usecols="A,B,H,I",
)

properties.head()

,Property Name,Portfolio Manager ID,Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Drinking Water Treatment & Distribution,325447
1,Island Water Treatment Plant,35000839,Drinking Water Treatment & Distribution,64196
2,W.H. Johnston Pumping Station,35000836,Drinking Water Treatment & Distribution,1744
3,West Toronto Pumping Station,35000837,Drinking Water Treatment & Distribution,7739
4,St. Albans Pumping Station,35000834,Drinking Water Treatment & Distribution,3240


## Create the gas meter dataframe

From the second worksheet, `Meter Entries - Gas`, we keep the requested columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `E`: `Meter Type`
- `G`: `Start Date`
- `H`: `End Date`
- `I`: `Usage/Quantity`
- `J`: `Usage Units`
- `K`: `Cost ($)`

This dataframe is named `gas_entries`.

In [4]:
gas_entries = pd.read_excel(
    file_path,
    sheet_name="Meter Entries - Gas",
    usecols="A,B,E,G,H,I,J,K",
)

gas_entries.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($)
0,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-01-01,2024-02-01,23839.31,cm (cubic meters),10317.96
1,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-02-01,2024-03-01,18735.80,cm (cubic meters),8181.12
2,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-03-01,2024-04-01,14082.81,cm (cubic meters),7422.93
3,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-04-01,2024-05-01,11537.50,cm (cubic meters),5674.99
4,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-05-01,2024-06-01,7784.48,cm (cubic meters),3675.73


## Create the electric meter dataframe

From the third worksheet, `Meter Entries - Electeric`, we keep the requested columns:
- `A`: `Property Name`
- `B`: `Portfolio Manager ID`
- `E`: `Meter Type`
- `G`: `Start Date`
- `H`: `End Date`
- `I`: `Usage/Quantity`
- `J`: `Usage Units`
- `K`: `Cost ($)`

This dataframe is named `electric_entries`.

In [5]:
electric_entries = pd.read_excel(
    file_path,
    sheet_name="Meter Entries - Electeric",
    usecols="A,B,E,G,H,I,J,K",
)

electric_entries.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($)
0,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-01-01,2024-02-01,3294191.84,kWh (thousand Watt-hours),335409.93
1,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-02-01,2024-03-01,3188060.05,kWh (thousand Watt-hours),300680.85
2,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-03-01,2024-04-01,2971166.22,kWh (thousand Watt-hours),273185.01
3,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-04-01,2024-05-01,2861591.94,kWh (thousand Watt-hours),255243.37
4,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-05-01,2024-06-01,3142472.63,kWh (thousand Watt-hours),285188.75


## Merge property information into the gas and electric dataframes

Now we attach the property-level information to both meter-entry datasets.

The merge uses both:
- `Property Name`
- `Portfolio Manager ID`

Using both columns makes the join explicit and keeps the property information aligned with the matching meter records.

In [6]:
merge_keys = ["Property Name", "Portfolio Manager ID"]

gas_with_properties = gas_entries.merge(
    properties,
    on=merge_keys,
    how="left",
)

gas_with_properties.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($),Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-01-01,2024-02-01,23839.31,cm (cubic meters),10317.96,Drinking Water Treatment & Distribution,325447
1,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-02-01,2024-03-01,18735.80,cm (cubic meters),8181.12,Drinking Water Treatment & Distribution,325447
2,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-03-01,2024-04-01,14082.81,cm (cubic meters),7422.93,Drinking Water Treatment & Distribution,325447
3,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-04-01,2024-05-01,11537.50,cm (cubic meters),5674.99,Drinking Water Treatment & Distribution,325447
4,F.J. Horgan Water Treatment Plant,35000838,Natural Gas,2024-05-01,2024-06-01,7784.48,cm (cubic meters),3675.73,Drinking Water Treatment & Distribution,325447


In [7]:
electric_with_properties = electric_entries.merge(
    properties,
    on=merge_keys,
    how="left",
)

electric_with_properties.head()

,Property Name,Portfolio Manager ID,Meter Type,Start Date,End Date,Usage/Quantity,Usage Units,Cost ($),Property Type - Self-Selected,Gross Floor Area
0,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-01-01,2024-02-01,3294191.84,kWh (thousand Watt-hours),335409.93,Drinking Water Treatment & Distribution,325447
1,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-02-01,2024-03-01,3188060.05,kWh (thousand Watt-hours),300680.85,Drinking Water Treatment & Distribution,325447
2,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-03-01,2024-04-01,2971166.22,kWh (thousand Watt-hours),273185.01,Drinking Water Treatment & Distribution,325447
3,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-04-01,2024-05-01,2861591.94,kWh (thousand Watt-hours),255243.37,Drinking Water Treatment & Distribution,325447
4,F.J. Horgan Water Treatment Plant,35000838,Electric - Grid,2024-05-01,2024-06-01,3142472.63,kWh (thousand Watt-hours),285188.75,Drinking Water Treatment & Distribution,325447


## Remove duplicate rows from the merged dataframes

After merging, some rows in gas_with_properties and electric_with_properties are duplicated.

This section removes fully duplicated rows from both merged dataframes and then prints the updated shapes so we can confirm the result.

In [9]:
gas_duplicates_before = gas_with_properties.duplicated().sum()
electric_duplicates_before = electric_with_properties.duplicated().sum()

gas_with_properties = gas_with_properties.drop_duplicates().reset_index(drop=True)
electric_with_properties = electric_with_properties.drop_duplicates().reset_index(drop=True)

print("Duplicate rows removed from gas_with_properties:", gas_duplicates_before)
print("Duplicate rows removed from electric_with_properties:", electric_duplicates_before)

print("\nNew shapes after removing duplicates:")
print("gas_with_properties:", gas_with_properties.shape)
print("electric_with_properties:", electric_with_properties.shape)

Duplicate rows removed from gas_with_properties: 3512
Duplicate rows removed from electric_with_properties: 4578

New shapes after removing duplicates:
gas_with_properties: (8522, 10)
electric_with_properties: (19832, 10)
